# Análisis de Precios de Vivienda mediante Modelos de Regresión

**Evaluación estadística y modelado predictivo**

Autor: RobertScience Data Consulting  
Proyecto: Práctica M33 – Estadística Avanzada & Regresión Lineal  
Herramienta: Python / Jupyter Notebook

## Índice / Flujo del Proyecto

1. Introducción y Objetivos  
2. Importación de librerías  
3. Carga del dataset  
4. Exploración inicial de datos  
5. Revisión de valores faltantes  
6. Estadística descriptiva y análisis de distribuciones  
7. Prueba de normalidad  
8. Análisis de correlaciones  
9. Selección de variables  
10. División Train/Test  
11. Construcción del modelo OLS  
12. Evaluación de multicolinealidad (VIF)  
13. Refinamiento del modelo  
14. Importancia de variables  
15. Evaluación del modelo  
16. Transformación logarítmica  
17. Validación gráfica  
18. Conclusiones

## Objetivos del proyecto

- Realizar un análisis exploratorio completo del dataset House Pricing.
- Identificar las variables que más influyen en el precio de venta de viviendas.
- Construir y optimizar un modelo de regresión lineal (OLS).
- Evaluar problemas de multicolinealidad mediante VIF.
- Mejorar el modelo mediante transformación logarítmica.
- Interpretar los resultados desde una perspectiva estadística y de negocio.

## Importación de librerías

In [ ]:
# ============================================
# Importación de librerías
# ============================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

sns.set(style="whitegrid")
%matplotlib inline

## Carga del dataset

El dataset **House Pricing** contiene información detallada de 1,460 propiedades residenciales, incluyendo características estructurales, calidad de construcción y precio de venta (`SalePrice`).

In [ ]:
# ============================================
# Carga del dataset
# ============================================
df = pd.read_csv('data/HousePricing.csv')
df.head()

## Exploración inicial de datos

En esta etapa se revisa la estructura general del dataset (dimensiones, tipos de datos y estadísticas básicas) para entender su composición.

In [ ]:
# ============================================
# Exploración inicial de datos
# ============================================
df.info()

In [ ]:
df.describe()

## Revisión de valores faltantes

Es fundamental detectar valores nulos ya que pueden afectar el rendimiento del modelo.

In [ ]:
# ============================================
# Revisión de valores faltantes
# ============================================
print(df.isnull().sum().sort_values(ascending=False).head(15))

## Selección inicial de variables para análisis descriptivo

Se seleccionan variables clave para realizar un primer análisis exploratorio.

In [ ]:
# ============================================
# Selección inicial de variables
# ============================================
variables = ['SalePrice', 'GrLivArea', '2ndFlrSF']
df_selected = df[variables]
df_selected.head()

## Análisis de distribuciones

Se visualizan los histogramas para entender la forma de distribución de las variables principales.

In [ ]:
# ============================================
# Análisis de distribuciones (Histogramas)
# ============================================
for col in variables:
    plt.figure(figsize=(8,5))
    sns.histplot(df_selected[col], kde=True)
    plt.title(f'Distribución de {col}')
    plt.xlabel(col)
    plt.ylabel('Frecuencia')
    plt.show()

## Prueba de normalidad

Se aplica la prueba de D'Agostino y Pearson para verificar si las variables siguen una distribución normal (requisito importante en regresión lineal).

In [ ]:
# ============================================
# Prueba de normalidad
# ============================================
for col in variables:
    stat, p = stats.normaltest(df_selected[col].dropna())
    print(f'\nVariable: {col}')
    print(f'Estadístico: {stat:.4f}')
    print(f'p-value: {p:.4f}')
    print('Conclusión: No sigue distribución normal' if p < 0.05 else 'Conclusión: Distribución aproximadamente normal')

## Análisis de correlaciones

Se busca identificar relaciones lineales entre las variables, especialmente con la variable objetivo `SalePrice`.

In [ ]:
# ============================================
# Análisis de correlaciones
# ============================================
df_numeric = df.select_dtypes(include=[np.number])
corr_matrix = df_numeric.corr()

plt.figure(figsize=(12,10))
sns.heatmap(corr_matrix, cmap='coolwarm', center=0, annot=False)
plt.title('Matriz de Correlación')
plt.show()

In [ ]:
corr_target = corr_matrix['SalePrice'].sort_values(ascending=False)
print(corr_target.head(15))

## Selección de variables para el modelo

Se seleccionan las variables con mayor correlación con `SalePrice` para construir el modelo predictivo.

In [ ]:
# ============================================
# Selección de variables para el modelo
# ============================================
top_vars = corr_target.index[1:11]
X = df[top_vars]
y = df['SalePrice']

## División del dataset (Train/Test)

Se divide el conjunto de datos para entrenar el modelo y evaluar su capacidad de generalización en datos no vistos.

In [ ]:
# ============================================
# División Train/Test
# ============================================
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

## Construcción del modelo OLS

Se construye un modelo de Regresión Lineal mediante Mínimos Cuadrados Ordinarios (OLS) utilizando statsmodels.

In [ ]:
# ============================================
# Modelo OLS Inicial
# ============================================
X_train_const = sm.add_constant(X_train)
model = sm.OLS(y_train, X_train_const).fit()
print(model.summary())

**Interpretación del modelo inicial**

- **R²**: Indica el porcentaje de variabilidad del precio explicado por el modelo.
- **R² ajustado**: Penaliza por número de variables.
- **Prob (F-statistic)**: Valida la significancia global del modelo.
- **p-values**: Indican la significancia individual de cada variable.

## Evaluación de multicolinealidad (VIF)

El Factor de Inflación de Varianza (VIF) detecta variables altamente correlacionadas entre sí.

In [ ]:
# ============================================
# Cálculo del VIF
# ============================================
X_vif = sm.add_constant(X_train)
vif_data = pd.DataFrame()
vif_data["Variable"] = X_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
print(vif_data)

**Interpretación del VIF**

Valores altos (> 5-10) indican multicolinealidad. Se procederá a eliminar variables redundantes.

## Refinamiento del modelo

Se elimina `GarageArea` por presentar multicolinealidad con `GarageCars`.

In [ ]:
# ============================================
# Modelo Refinado
# ============================================
X_refined = X.drop(columns=['GarageArea'])
X_train_refined = X_train[X_refined.columns]
X_test_refined = X_test[X_refined.columns]

X_train_refined_const = sm.add_constant(X_train_refined)
model_refined = sm.OLS(y_train, X_train_refined_const).fit()
print(model_refined.summary())

## Importancia de variables

Se analizan los coeficientes para determinar el impacto relativo de cada variable en el precio.

In [ ]:
# ============================================
# Importancia de variables
# ============================================
coef_df = pd.DataFrame({
    'Variable': model_refined.params.index,
    'Coeficiente': model_refined.params.values
})
print(coef_df.sort_values(by='Coeficiente', ascending=False))

## Evaluación del modelo

Se calculan métricas en el conjunto de prueba para evaluar el desempeño predictivo.

In [ ]:
# ============================================
# Evaluación del modelo
# ============================================
X_test_refined_const = sm.add_constant(X_test_refined)
y_pred = model_refined.predict(X_test_refined_const)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"RMSE (test): {rmse:.2f}")
print(f"R² (test): {r2:.4f}")
print(f"MAE: {mae:.2f}")

## Transformación logarítmica

Dado que `SalePrice` presenta asimetría positiva, se aplica logaritmo para mejorar el ajuste y estabilizar la varianza.

In [ ]:
# ============================================
# Transformación Logarítmica
# ============================================
y_train_log = np.log(y_train)
y_test_log = np.log(y_test)

model_log = sm.OLS(y_train_log, X_train_refined_const).fit()
print(model_log.summary())

y_pred_log = model_log.predict(X_test_refined_const)
y_pred_exp = np.exp(y_pred_log)

rmse_log = np.sqrt(mean_squared_error(y_test, y_pred_exp))
r2_log = r2_score(y_test, y_pred_exp)
print(f"RMSE LOG (test): {rmse_log:.2f}")
print(f"R² LOG (test): {r2_log:.4f}")

**Comparación**: El modelo con transformación logarítmica muestra mejor desempeño (mayor R² y menor RMSE).

## Validación gráfica

Se generan gráficos para validar visualmente los supuestos del modelo y la calidad de las predicciones.

In [ ]:
# ============================================
# Validación gráfica
# ============================================
plt.figure(figsize=(8,6))
plt.scatter(y_test, y_pred)
plt.xlabel("Valores reales")
plt.ylabel("Predicciones")
plt.title("Valores reales vs predicción")
plt.show()

residuals = y_test - y_pred
plt.figure(figsize=(8,5))
sns.histplot(residuals, kde=True)
plt.title("Distribución de residuales")
plt.show()

plt.figure(figsize=(8,5))
plt.scatter(y_pred, residuals)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel("Predicciones")
plt.ylabel("Residuales")
plt.title("Residuales vs Predicción")
plt.show()

## Conclusiones

El análisis exploratorio reveló que `SalePrice` y otras variables clave no siguen una distribución normal y presentan asimetría positiva. Mediante el análisis de correlación se identificaron las variables más influyentes: `OverallQual`, `GrLivArea`, `GarageCars`, `TotalBsmtSF`, entre otras.

Se construyó un modelo de regresión lineal OLS que alcanzó un **R² de aproximadamente 0.76** en el conjunto de prueba. Tras detectar multicolinealidad mediante VIF y refinar el modelo, se obtuvo una versión más estable. La transformación logarítmica de la variable objetivo mejoró significativamente el ajuste (**R² ≈ 0.87**), reduciendo el error de predicción.

Los gráficos de validación confirman que el modelo captura adecuadamente la relación entre las características de las viviendas y su precio, con residuales distribuidos de forma razonable.

**Valor de negocio**: Este modelo puede utilizarse como base para sistemas de valuación automática (AVM), detección de oportunidades inmobiliarias y soporte en la toma de decisiones de inversión en el sector PropTech.

Como mejora futura se recomienda explorar técnicas de regularización (Ridge/Lasso) y modelos más avanzados (Random Forest, Gradient Boosting).